# Clouddex — train the cloud classifier on Google Colab (free GPU)

This notebook trains the MobileNetV2 cloud-genus classifier and exports it to
**TensorFlow.js** for the Clouddex PWA — entirely in the cloud, no local install.

**How to use**
1. Runtime → **Change runtime type** → Hardware accelerator = **GPU** → Save.
2. Run the cells top to bottom.
3. Provide the **CCSN** dataset when prompted (upload a `.zip`, mount Drive, or a URL).
4. The last cell downloads `clouddex_model.zip`. Unzip it into the repo's
   `public/model/` folder, commit, and push — the live site redeploys with the
   real model.

> Preprocessing here (224×224, MobileNetV2 `[-1,1]`) matches `src/ml/preprocess.ts`
> in the app. Don't change one without the other.

In [ ]:
# 1. Sanity check: TensorFlow + GPU
import tensorflow as tf
print("TensorFlow", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "NONE — set Runtime > Change runtime type > GPU")

## 2. Get the CCSN dataset

Pick **one** option below. Each class must end up as its own folder under
`DATA_DIR` (either CCSN's 2-letter codes `Ci/ Cu/ …` or full names — the next
cell normalizes them).

In [ ]:
# --- Option A (default): upload a .zip of the dataset ---
import os, zipfile
from google.colab import files

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

uploaded = files.upload()  # choose your CCSN .zip
for name in uploaded:
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(name) as z:
            z.extractall(DATA_DIR)
        print("extracted", name)

# If everything nested under a single top folder, descend into it.
subdirs = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
if len(subdirs) == 1:
    inner = os.path.join(DATA_DIR, subdirs[0])
    if any(os.path.isdir(os.path.join(inner, d)) for d in os.listdir(inner)):
        DATA_DIR = inner
print("DATA_DIR =", DATA_DIR)
print("classes:", sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))))

In [ ]:
# --- Option B: mount Google Drive (put the dataset in your Drive first) ---
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_DIR = "/content/drive/MyDrive/CCSN"   # adjust to your folder

# --- Option C: download from a direct URL (a .zip) ---
# import os, zipfile, urllib.request
# DATA_DIR = "/content/data"; os.makedirs(DATA_DIR, exist_ok=True)
# urllib.request.urlretrieve("https://example.com/CCSN.zip", "/content/ccsn.zip")
# with zipfile.ZipFile("/content/ccsn.zip") as z: z.extractall(DATA_DIR)

## 3. Prepare the dataset (structure + cleaning)

Handles the messy reality of real dataset zips, especially **Mac-made zips**
(which add a `__MACOSX` folder full of `._` AppleDouble files and often nest the
images one level down). This cell:

- deletes `__MACOSX` junk,
- descends to the real folder that holds the class subfolders (reassigns `DATA_DIR`),
- renames CCSN's 2-letter codes (`Ci`, `Cu`, …) to the app's full class ids,
- removes `._` files and any image TensorFlow can't decode (re-saving salvageable ones).

In [ ]:
# 3. Prepare: fix structure, rename classes, clean undecodable files
import os, shutil, tensorflow as tf
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = False

shutil.rmtree(os.path.join(DATA_DIR, "__MACOSX"), ignore_errors=True)

def _has_images(d):
    try:
        return any(f.lower().endswith((".jpg", ".jpeg", ".png")) for f in os.listdir(d))
    except Exception:
        return False

def _find_class_root(base):
    cur = base
    for _ in range(6):
        subs = [d for d in os.listdir(cur)
                if os.path.isdir(os.path.join(cur, d)) and not d.startswith("__")]
        if len(subs) >= 2 and any(_has_images(os.path.join(cur, s)) for s in subs):
            return cur
        if len(subs) == 1:
            cur = os.path.join(cur, subs[0]); continue
        return cur
    return cur

DATA_DIR = _find_class_root(DATA_DIR)
print("DATA_DIR =", DATA_DIR)

MAP = {"Ci": "cirrus", "Cs": "cirrostratus", "Cc": "cirrocumulus",
       "Ac": "altocumulus", "As": "altostratus", "Cu": "cumulus",
       "Cb": "cumulonimbus", "Ns": "nimbostratus", "Sc": "stratocumulus",
       "St": "stratus", "Ct": "contrail"}
for code_, full in MAP.items():
    s, d = os.path.join(DATA_DIR, code_), os.path.join(DATA_DIR, full)
    if os.path.isdir(s) and not os.path.isdir(d):
        os.rename(s, d)

classes = sorted(d for d in os.listdir(DATA_DIR)
                 if os.path.isdir(os.path.join(DATA_DIR, d)) and not d.startswith("__"))
removed = 0
for c in classes:
    cdir = os.path.join(DATA_DIR, c)
    for fn in list(os.listdir(cdir)):
        fp = os.path.join(cdir, fn)
        if fn.startswith("._") or not os.path.isfile(fp):
            if os.path.isfile(fp):
                os.remove(fp); removed += 1
            continue
        try:
            tf.image.decode_image(tf.io.read_file(fp), expand_animations=False)
        except Exception:
            try:
                Image.open(fp).convert("RGB").save(fp, "JPEG", quality=92)
            except Exception:
                os.remove(fp); removed += 1

print("classes:", classes)
print("counts:", {c: len(os.listdir(os.path.join(DATA_DIR, c))) for c in classes})
print("removed junk/undecodable:", removed)

In [ ]:
# 4. Build train/val datasets + class weights (CCSN is imbalanced)
import os, tensorflow as tf, keras
IMG_SIZE, BATCH = 224, 32

train_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="training", seed=1337,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, label_mode="categorical")
val_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="validation", seed=1337,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, label_mode="categorical")

class_names = train_ds.class_names
print("MODEL OUTPUT ORDER (this becomes labels.json):", class_names)

counts = [len(os.listdir(os.path.join(DATA_DIR, c))) for c in class_names]
total, n = sum(counts), len(class_names)
class_weight = {i: (total / (n * c) if c else 1.0) for i, c in enumerate(counts)}

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [ ]:
# 5. MobileNetV2 transfer-learning model
import keras
from keras import layers, models
from keras.applications import MobileNetV2
from keras.applications.mobilenet_v2 import preprocess_input

augment = models.Sequential([
    layers.RandomFlip("horizontal"), layers.RandomRotation(0.05),
    layers.RandomZoom(0.1), layers.RandomContrast(0.1)], name="augment")

base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False,
                   weights="imagenet")
base.trainable = False

inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))
x = augment(inputs)
x = preprocess_input(x)          # -> [-1, 1], matches src/ml/preprocess.ts
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)
model = models.Model(inputs, outputs)
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="categorical_crossentropy", metrics=["accuracy"])

In [ ]:
# 6. Train the head (feature extraction)
model.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=class_weight)

In [ ]:
# 7. Fine-tune the top of the backbone at a low LR
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss="categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=8, class_weight=class_weight)

In [ ]:
# 8. Export a SavedModel + labels.json (in the model's true output order)
import json
model.export("/content/saved_model")
with open("/content/labels.json", "w") as f:
    json.dump({"labels": class_names}, f, indent=2)
print("Saved /content/saved_model and labels:", class_names)

## 9. Install the TF.js converter — then RESTART the runtime

`tensorflowjs` pulls dependency versions that conflict with the TensorFlow
already loaded in this session, which makes the converter fail. So install it,
then run it in a **fresh kernel**: after this cell, do **Runtime → Restart
session** (this does *not* delete `/content` — your trained model stays on disk
at `/content/saved_model`) and continue from cell 10.

In [ ]:
# 9. Install the converter, then RESTART (Runtime > Restart session)
!pip install -q tensorflowjs
print("Installed. NOW do: Runtime > Restart session, then run cell 10.")
print("Your model persists at /content/saved_model — a restart won't lose it.")

## 10. Convert → package → download  (run AFTER restarting)

Runs the converter in the clean kernel and prints its full output. If it fails,
paste the error — or use the **local fallback** in the last cell (download the
SavedModel and convert with the repo's `convert.sh`, which already works).

In [ ]:
# 10. Convert -> package -> download
import subprocess, os, shutil
os.makedirs("/content/web_model", exist_ok=True)
res = subprocess.run(
    ["tensorflowjs_converter", "--input_format=tf_saved_model",
     "--output_format=tfjs_graph_model", "--signature_name=serving_default",
     "--saved_model_tags=serve", "/content/saved_model", "/content/web_model"],
    capture_output=True, text=True)
print(res.stdout[-1500:])
print("STDERR:\n", res.stderr[-3000:])
assert res.returncode == 0, "conversion failed — paste the STDERR above"
shutil.copy("/content/labels.json", "/content/web_model/labels.json")
shutil.make_archive("/content/clouddex_model", "zip", "/content/web_model")
print("web_model:", os.listdir("/content/web_model"))
from google.colab import files
files.download("/content/clouddex_model.zip")

### Fallback: convert on your own machine

If the Colab converter keeps failing, skip it: download the raw SavedModel here
and convert locally with the repo's already-working converter.

In [ ]:
# Fallback: download the raw SavedModel to convert locally
import shutil
from google.colab import files
shutil.make_archive("/content/saved_model", "zip", "/content/saved_model")
files.download("/content/saved_model.zip")
# Then, locally from the repo root:
#   unzip ~/Downloads/saved_model.zip -d training/saved_model
#   cd training && ./.venv/bin/tensorflowjs_converter \
#       --input_format=tf_saved_model --output_format=tfjs_graph_model \
#       --signature_name=serving_default --saved_model_tags=serve \
#       saved_model ../public/model
#   # labels.json: keep the repo's, or copy the one trained alongside the model

## 11. Install the model in the app

1. Unzip `clouddex_model.zip` — you'll get `model.json`, one or more `*.bin`
   weight shards, and `labels.json`.
2. Put all of those into the repo's **`public/model/`** folder (replacing the
   placeholder `labels.json`).
3. Commit and push:
   ```bash
   git add public/model && git commit -m "Add trained model" && git push
   ```
4. The GitHub Actions deploy runs automatically; the live site loses its
   **“demo model”** badge and starts making real predictions.